# Generative Adversarial Network for Medical Imaging: Chest X-Ray Generation

## Learning Objectives
By the end of this notebook, you will:
1. Understand the mathematical foundations of Generative Adversarial Networks (GANs)
2. Implement a GAN with U-Net generator and PatchGAN discriminator for medical image generation
3. Master adversarial training dynamics between generator and discriminator
4. Evaluate generative models using appropriate metrics
5. Apply GANs to real medical imaging data (chest X-rays)

---

## 1. Introduction to Generative Adversarial Networks

### What is a GAN?

A **Generative Adversarial Network (GAN)** is a generative model consisting of two neural networks competing against each other in a zero-sum game. Unlike VAEs, GANs:

- Use **adversarial training** where a generator creates fake images and a discriminator tries to distinguish them from real images
- Can **generate high-quality samples** through this adversarial process
- Don't require explicit likelihood computation

### Key Components:

1. **Generator**: Maps latent noise z → generated image x̂
2. **Discriminator**: Classifies images as real or fake
3. **Adversarial Loss**: MinMax game between generator and discriminator
4. **PatchGAN Discriminator**: Evaluates image patches for local realism


## 2. Setup and Imports


In [ ]:
# Install required packages if needed
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Install required packages
packages = ['torch', 'torchvision', 'numpy', 'matplotlib', 'Pillow', 'tqdm', 'scikit-learn', 'seaborn']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        install_package(package)


In [ ]:
# Core imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

# Utilities
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
import time
import warnings
import zipfile
import urllib.request
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
from typing import Tuple, Optional

warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")


## 3. Device Configuration


In [ ]:
def get_device() -> torch.device:
    """
    Automatically detect and return the best available device.
    
    Returns:
        torch.device: CUDA device if available, MPS for Mac Silicon, else CPU
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
        print(f"   Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")
    else:
        device = torch.device("cpu")
        print("Using CPU")
    
    return device

device = get_device()

# Set random seeds for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


## 4. Data Loading and Preprocessing

### Downloading Chest X-Ray Dataset

We'll use the same chest X-ray dataset as the VAE notebook.


In [ ]:
def download_chest_xray_data(data_dir='./data'):
    """
    Download a real chest X-ray dataset.
    Using the NIH Chest X-ray14 dataset subset for educational purposes.
    """
    os.makedirs(data_dir, exist_ok=True)
    chest_xray_dir = os.path.join(data_dir, "chest_xray_images")
    
    if not os.path.exists(chest_xray_dir):
        print("Setting up chest X-ray dataset...")
        print("Note: For educational purposes, we'll use a sample dataset.")
        
        try:
            sample_url = "https://github.com/ieee8023/covid-chestxray-dataset/archive/master.zip"
            zip_path = os.path.join(data_dir, "sample_chest_xray.zip")
            
            print("Downloading sample chest X-ray images...")
            urllib.request.urlretrieve(sample_url, zip_path)
            
            print("Extracting dataset...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            
            extracted_dir = os.path.join(data_dir, "covid-chestxray-dataset-master")
            if os.path.exists(extracted_dir):
                for root, dirs, files in os.walk(extracted_dir):
                    for file in files:
                        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                            os.makedirs(chest_xray_dir, exist_ok=True)
                            src_path = os.path.join(root, file)
                            dst_path = os.path.join(chest_xray_dir, file)
                            import shutil
                            shutil.copy2(src_path, dst_path)
            
            print("Real chest X-ray dataset ready!")
            
        except Exception as e:
            print(f"Could not download dataset: {e}")
            print("Please manually download chest X-ray images")
    else:
        print("Real chest X-ray dataset already exists!")
    
    return chest_xray_dir

def create_synthetic_chest_xray_data(data_dir, num_samples=1000):
    """Create synthetic chest X-ray-like images for demonstration."""
    img_dir = os.path.join(data_dir, "chest_xray_images")
    os.makedirs(img_dir, exist_ok=True)
    
    print(f"Creating {num_samples} synthetic chest X-ray images...")
    
    for i in tqdm(range(num_samples), desc="Generating images"):
        img = np.zeros((256, 256), dtype=np.uint8)
        center_x, center_y = 128, 128
        
        for x in range(256):
            for y in range(256):
                dist_left = np.sqrt((x - (center_x - 40))**2 + (y - center_y)**2)
                dist_right = np.sqrt((x - (center_x + 40))**2 + (y - center_y)**2)
                
                if dist_left < 60:
                    img[y, x] = max(img[y, x], int(180 - dist_left * 2))
                if dist_right < 60:
                    img[y, x] = max(img[y, x], int(180 - dist_right * 2))
        
        noise = np.random.normal(0, 10, (256, 256))
        img = np.clip(img + noise, 0, 255).astype(np.uint8)
        
        if i % 3 == 0:
            x, y = np.random.randint(50, 200, 2)
            radius = np.random.randint(5, 20)
            for dx in range(-radius, radius):
                for dy in range(-radius, radius):
                    if dx**2 + dy**2 < radius**2:
                        if 0 <= x+dx < 256 and 0 <= y+dy < 256:
                            img[y+dy, x+dx] = min(255, img[y+dy, x+dx] + 50)
        
        Image.fromarray(img, mode='L').save(os.path.join(img_dir, f"chest_{i:04d}.png"))
    
    print("Synthetic data created!")

# Download data
data_path = download_chest_xray_data('./data')


### Custom Dataset Class


In [ ]:
class ChestXRayDataset(Dataset):
    """
    Custom Dataset for loading chest X-ray images.
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = [f for f in os.listdir(root_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.images[idx])
        image = Image.open(img_path).convert('L')
        
        if self.transform:
            image = self.transform(image)
        
        return image

# Define transformations
img_size = 128

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Create dataset
dataset = ChestXRayDataset(data_path, transform=transform)

# Split into train and validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Dataset Statistics:")
print(f"   Total samples: {len(dataset)}")
print(f"   Training samples: {train_size}")
print(f"   Validation samples: {val_size}")
print(f"   Batch size: {batch_size}")
print(f"   Image dimensions: {img_size}x{img_size}")


## 5. GAN Architecture: Generator and Discriminator

### Understanding the Architecture

The GAN consists of:
1. **U-Net Generator**: Maps latent noise z → generated chest X-ray image
2. **PatchGAN Discriminator**: Evaluates N×N patches to classify real vs fake
3. **Adversarial Training**: Generator tries to fool discriminator, discriminator tries to detect fakes


In [ ]:
class ConvBlock(nn.Module):
    """
    Basic convolutional block with BatchNorm and ReLU.
    Building block for U-Net architecture.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super(ConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size, padding=padding)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x


In [ ]:
class Generator_UNet(nn.Module):
    """
    U-Net style Generator for GAN.
    Maps latent noise vector to chest X-ray image.
    """
    def __init__(self, latent_dim=128, in_channels=1):
        super(Generator_UNet, self).__init__()
        self.latent_dim = latent_dim
        
        # Calculate the size after all pooling operations
        self.flatten_size = 256 * (img_size // 16) * (img_size // 16)
        
        # Fully connected layer from latent to feature map
        self.fc = nn.Linear(latent_dim, self.flatten_size)
        
        # Upsampling layers
        self.upconv1 = nn.ConvTranspose2d(256, 256, kernel_size=2, stride=2)
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.upconv3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.upconv4 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        
        # Decoder convolutional blocks
        self.dec_conv1 = ConvBlock(256, 128)
        self.dec_conv2 = ConvBlock(128, 64)
        self.dec_conv3 = ConvBlock(64, 32)
        self.dec_conv4 = ConvBlock(32, 16)
        
        # Final layer
        self.final_conv = nn.Conv2d(16, in_channels, kernel_size=1)
        
    def forward(self, z):
        # Project and reshape
        x = self.fc(z)
        x = x.view(x.size(0), 256, img_size // 16, img_size // 16)
        
        # Decode
        x = self.upconv1(x)      # 8x8 -> 16x16, channels: 256->256
        x = self.dec_conv1(x)     # 16x16 -> 16x16, channels: 256->128
        
        x = self.upconv2(x)      # 16x16 -> 32x32, channels: 128->128
        x = self.dec_conv2(x)     # 32x32 -> 32x32, channels: 128->64
        
        x = self.upconv3(x)      # 32x32 -> 64x64, channels: 64->64
        x = self.dec_conv3(x)     # 64x64 -> 64x64, channels: 64->32
        
        x = self.upconv4(x)      # 64x64 -> 128x128, channels: 32->32
        x = self.dec_conv4(x)     # 128x128 -> 128x128, channels: 32->16
        
        x = self.final_conv(x)   # channels: 16->1
        x = torch.tanh(x)        # Output in range [-1, 1]
        
        return x


In [ ]:
class PatchGAN_Discriminator(nn.Module):
    """
    PatchGAN Discriminator that classifies N×N patches as real or fake.
    This provides more detailed feedback to the generator than a single classification.
    """
    def __init__(self, in_channels=1):
        super(PatchGAN_Discriminator, self).__init__()
        
        def discriminator_block(in_filters, out_filters, normalization=True):
            """Returns downsampling layers of each discriminator block"""
            layers = [nn.Conv2d(in_filters, out_filters, 4, stride=2, padding=1)]
            if normalization:
                layers.append(nn.BatchNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        
        self.model = nn.Sequential(
            # Input: 128x128x1
            *discriminator_block(in_channels, 64, normalization=False),  # 64x64x64
            *discriminator_block(64, 128),   # 32x32x128
            *discriminator_block(128, 256),  # 16x16x256
            *discriminator_block(256, 512),  # 8x8x512
            nn.Conv2d(512, 1, 4, padding=1)  # 8x8x1 (patch output)
        )
    
    def forward(self, img):
        return self.model(img)


In [ ]:
# Initialize models
latent_dim = 128
generator = Generator_UNet(latent_dim=latent_dim, in_channels=1).to(device)
discriminator = PatchGAN_Discriminator(in_channels=1).to(device)

# Count parameters
gen_params = sum(p.numel() for p in generator.parameters())
disc_params = sum(p.numel() for p in discriminator.parameters())

print(f"Model Architecture:")
print(f"   Generator parameters: {gen_params:,}")
print(f"   Discriminator parameters: {disc_params:,}")
print(f"   Total parameters: {gen_params + disc_params:,}")
print(f"   Latent dimension: {latent_dim}")


## 6. Adversarial Loss Function

### Understanding GAN Loss

The GAN loss consists of:
1. **Discriminator Loss**: Maximize ability to distinguish real from fake
2. **Generator Loss**: Minimize discriminator's ability to detect fakes

**Mathematical formulation:**

Discriminator: $\max_D \mathbb{E}_{x}[\log D(x)] + \mathbb{E}_{z}[\log(1 - D(G(z)))]$

Generator: $\min_G \mathbb{E}_{z}[\log(1 - D(G(z)))]$ or equivalently $\max_G \mathbb{E}_{z}[\log D(G(z))]$


In [ ]:
# Loss function - Binary Cross Entropy for adversarial training
adversarial_loss = nn.BCEWithLogitsLoss()

# Optimizers
lr_gen = 0.0002
lr_disc = 0.0002
beta1 = 0.5

optimizer_G = optim.Adam(generator.parameters(), lr=lr_gen, betas=(beta1, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr_disc, betas=(beta1, 0.999))

print(f"Training Configuration:")
print(f"   Loss Function: Binary Cross Entropy (adversarial)")
print(f"   Generator LR: {lr_gen}")
print(f"   Discriminator LR: {lr_disc}")
print(f"   Beta1: {beta1}")


## 7. Adversarial Training Loop


In [ ]:
def train_gan_epoch(generator, discriminator, dataloader, optimizer_G, optimizer_D, adversarial_loss, device):
    """
    Train GAN for one epoch with adversarial training.
    """
    generator.train()
    discriminator.train()
    
    total_d_loss = 0
    total_g_loss = 0
    
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    
    for batch_idx, real_imgs in enumerate(progress_bar):
        batch_size = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        
        # Adversarial ground truths
        valid = torch.ones(batch_size, 1, 8, 8).to(device)  # PatchGAN output size
        fake = torch.zeros(batch_size, 1, 8, 8).to(device)
        
        # ---------------------
        #  Train Discriminator
        # ---------------------
        optimizer_D.zero_grad()
        
        # Real images
        real_pred = discriminator(real_imgs)
        d_real_loss = adversarial_loss(real_pred, valid)
        
        # Fake images
        z = torch.randn(batch_size, latent_dim).to(device)
        fake_imgs = generator(z)
        fake_pred = discriminator(fake_imgs.detach())
        d_fake_loss = adversarial_loss(fake_pred, fake)
        
        # Total discriminator loss
        d_loss = (d_real_loss + d_fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()
        
        # -----------------
        #  Train Generator
        # -----------------
        optimizer_G.zero_grad()
        
        # Generate fake images
        z = torch.randn(batch_size, latent_dim).to(device)
        gen_imgs = generator(z)
        
        # Generator wants discriminator to think images are real
        gen_pred = discriminator(gen_imgs)
        g_loss = adversarial_loss(gen_pred, valid)
        
        g_loss.backward()
        optimizer_G.step()
        
        # Accumulate losses
        total_d_loss += d_loss.item()
        total_g_loss += g_loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'D_loss': f'{d_loss.item():.4f}',
            'G_loss': f'{g_loss.item():.4f}'
        })
    
    avg_d_loss = total_d_loss / len(dataloader)
    avg_g_loss = total_g_loss / len(dataloader)
    
    return avg_d_loss, avg_g_loss


In [ ]:
# Training parameters
num_epochs = 50
history = {'d_loss': [], 'g_loss': []}
best_g_loss = float('inf')

print("Starting GAN Training...")
print("="*50)

for epoch in range(num_epochs):
    start_time = time.time()
    
    # Train for one epoch
    d_loss, g_loss = train_gan_epoch(generator, discriminator, train_loader, 
                                      optimizer_G, optimizer_D, adversarial_loss, device)
    
    # Save history
    history['d_loss'].append(d_loss)
    history['g_loss'].append(g_loss)
    
    # Save best model
    if g_loss < best_g_loss:
        best_g_loss = g_loss
        torch.save({
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'optimizer_D_state_dict': optimizer_D.state_dict(),
            'g_loss': g_loss,
            'd_loss': d_loss,
        }, 'best_gan_model.pth')
    
    # Print epoch results
    epoch_time = time.time() - start_time
    print(f"Epoch [{epoch+1}/{num_epochs}] ({epoch_time:.2f}s)")
    print(f"  D Loss: {d_loss:.4f} | G Loss: {g_loss:.4f}")
    print("-"*50)

print("\nTraining completed!")
print(f"Best generator loss: {best_g_loss:.4f}")


## 8. Training Visualization


In [ ]:
def plot_training_history(history):
    """
    Visualize training losses over epochs.
    """
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    
    ax.plot(history['d_loss'], label='Discriminator Loss', linewidth=2)
    ax.plot(history['g_loss'], label='Generator Loss', linewidth=2)
    ax.set_title('GAN Training Loss', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plot training history
plot_training_history(history)


## 9. Image Generation

### Load Best Model


In [ ]:
# Load the best model
checkpoint = torch.load('best_gan_model.pth', map_location=device)
generator.load_state_dict(checkpoint['generator_state_dict'])
generator.eval()
print(f"Loaded best generator from epoch {checkpoint['epoch']+1}")


### Generate Samples


In [ ]:
def generate_samples(generator, num_samples, device):
    """
    Generate new samples using the trained generator.
    """
    generator.eval()
    
    with torch.no_grad():
        z = torch.randn(num_samples, latent_dim).to(device)
        samples = generator(z)
        samples = (samples + 1) / 2  # Denormalize
    
    return samples

def visualize_generated_samples(samples, rows=2, cols=4):
    """
    Visualize generated samples in a grid.
    """
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2, rows*2))
    axes = axes.flatten()
    
    for i in range(min(len(samples), rows*cols)):
        axes[i].imshow(samples[i].cpu().squeeze(), cmap='gray')
        axes[i].axis('off')
    
    plt.suptitle('GAN Generated Chest X-ray Images', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Generate and visualize samples
generated_samples = generate_samples(generator, num_samples=8, device=device)
visualize_generated_samples(generated_samples)
